![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







### Copyright notice

> <p><small><small>Copyright 2025 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

# Manipulation in The Playground! <a href="https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/manipulation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a>

In this notebook, we'll walk through a couple manipulation environments available in MuJoCo Playground.

**A Colab runtime with GPU acceleration is required.** If you're using a CPU-only runtime, you can switch using the menu "Runtime > Change runtime type".


In [ ]:
# # #@title Install pre-requisites
!pip install mujoco
!pip install mujoco_mjx
!pip install brax

# !pip install --break-system-packages mediapy -q

In [ ]:
!pip install mediapy

In [ ]:
# @title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

In [ ]:
# @title Import packages for plotting and creating graphics
import json
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
# print("Installing mediapy:")
# !command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
# !pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [ ]:
# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
from jax import numpy as jp
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

In [ ]:
#@title Install MuJoCo Playground
# !pip install playground

In [ ]:
#@title Import The Playground

from mujoco_playground import wrapper
from mujoco_playground import registry
from mujoco_playground import registry

In [ ]:
print(registry.__file__)

# Manipulation

MuJoCo Playground contains several manipulation environments (all listed below after running the command).

In [ ]:
registry.manipulation.ALL_ENVS

# Franka Emika Panda

Let's start off with the simplest environment, simply picking up a cube with the Franka Emika Panda.

In [ ]:
# env_name = 'PandaPickCubeOrientation'
# env = registry.load(env_name)
# env_cfg = registry.get_default_config(env_name)

In [ ]:
env_name = 'DualUR5eBoxlift'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)

In [ ]:
env_cfg
env_cfg.sim_dt

## Train Policy

Let's train the pick cube policy and visualize rollouts. The policy takes roughly 3 minutes to train on an RTX 4090.

In [ ]:
CHECKPOINT_IDX = 0
TRAINING = True

In [ ]:
from mujoco_playground.config import manipulation_params
ppo_params = manipulation_params.brax_ppo_config(env_name)
ppo_params

In [ ]:
from mujoco_playground.config import manipulation_params
ppo_params = manipulation_params.brax_ppo_config(env_name)

# # Remove asymmetric observation keys since this env uses flat obs
# if "policy_obs_key" in ppo_params:
#     del ppo_params["policy_obs_key"]
# if "value_obs_key" in ppo_params:
#     del ppo_params["value_obs_key"]

print("Updated ppo_params - removed obs keys")
print(f"policy_obs_key: {ppo_params.get('policy_obs_key')}")
print(f"value_obs_key: {ppo_params.get('value_obs_key')}")

In [ ]:


# 🔍 DEBUG HERE (before training)
rng = jax.random.PRNGKey(0)
state = env.reset(rng)
data = state.data

print("qpos:", data.qpos.shape, data.qpos.size)
print("qvel:", data.qvel.shape, data.qvel.size)

# target_id = env._mj_model.body_mocapid[
#     env._mj_model.body(name='target_0').id
# ]

# print("target_id:", target_id)
print("mocap_pos:", data.mocap_pos.shape)

print("OBS: state", state.obs['state'].shape)
print("OBS: privileged state", state.obs['privileged_state'].shape)


In [ ]:


x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory_kwargs = dict(ppo_params.network_factory)
  # Remove asymmetric obs keys — DualUR5eBoxlift returns flat obs, not a dict
  network_factory_kwargs.pop("policy_obs_key", None)
  network_factory_kwargs.pop("value_obs_key", None)
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **network_factory_kwargs
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    seed=1,
    save_checkpoint_path=os.path.abspath(f"./checkpoints/run_{CHECKPOINT_IDX}"),
)


In [ ]:
dict(ppo_training_params)

In [ ]:
# # Diagnostic: intercept network_factory to see what ppo.train passes
# _original_nf = network_factory
# def debug_network_factory(obs_size, action_size, **kwargs):
#     print(f"[DEBUG] network_factory called with:")
#     print(f"  obs_size = {obs_size} (type={type(obs_size)})")
#     print(f"  action_size = {action_size}")
#     print(f"  kwargs keys = {list(kwargs.keys())}")
#     return _original_nf(obs_size, action_size, **kwargs)

# # Temporarily replace network_factory in train_fn
# import copy
# train_fn_debug = functools.partial(
#     ppo.train,
#     **{k: v for k, v in ppo_training_params.items()},
#     network_factory=debug_network_factory,
#     progress_fn=progress,
#     seed=1,
# )
# make_inference_fn, params, metrics = train_fn_debug(
#     environment=env,
#     wrap_env_fn=wrapper.wrap_for_brax_training,
# )


### PPO DUAL ARM

In [ ]:
print("\n" + "=" * 50)
print(state.obs['state'])
print("\n" + "=" * 50)
print(state.obs['privileged_state'])

In [ ]:
def obs_debug_full(obs_dict, env):
    """
    Automatic debugger for observation dict WITHOUT hardcoded layout.
    Infers structure based on sizes and known env properties.
    """

    def debug_single(obs, name):
        print("\n" + "=" * 70)
        print(f"DEBUGGING: {name}")
        print("=" * 70)

        idx = 0
        total = obs.shape[0]

        def take(n, label):
            nonlocal idx
            if idx + n > total:
                return None
            val = obs[idx:idx+n]
            print(f"{label:30s} [{idx}:{idx+n}] → {val}")
            idx += n
            return val

        # -----------------------------
        # Known anchors
        # -----------------------------
        num_dof = env.num_dof
        nv = env._mjx_model.nv

        # -----------------------------
        # 1. Joint positions (always first)
        # -----------------------------
        joint_pos = take(num_dof, "joint_pos")

        # -----------------------------
        # 2. Try detect next block
        # -----------------------------
        remaining = total - idx

        # Heuristic: if next chunk is 7 → ball_pose
        if remaining >= 7:
            # Peek ahead safely
            candidate = obs[idx:idx+7]

            # Heuristic: quaternion norm ~1
            quat = candidate[3:]
            if jp.isfinite(quat).all() and jp.abs(jp.linalg.norm(quat) - 1.0) < 0.2:
                ball_pose = take(7, "ball_pose")
                print(f"  ├─ ball_pos  → {ball_pose[:3]}")
                print(f"  └─ ball_quat → {ball_pose[3:]}")

        # -----------------------------
        # 3. Velocities (always nv)
        # -----------------------------
        qvel = take(nv, "qvel")

        if qvel is not None:
            joint_vel = qvel[env._joint_mask_vel]

            ball_body_id = env._mj_model.body(name="ball").id
            ball_qvel_idx = env._mj_model.body_dofadr[ball_body_id]

            ball_vel = qvel[ball_qvel_idx:ball_qvel_idx+3]
            ball_ang_vel = qvel[ball_qvel_idx+3:ball_qvel_idx+6]

            print(f"  ├─ joint_vel     → {joint_vel}")
            print(f"  ├─ ball_vel      → {ball_vel}")
            print(f"  └─ ball_ang_vel  → {ball_ang_vel}")

        # -----------------------------
        # 4. Remaining blocks (auto detect)
        # -----------------------------
        while idx < total:
            remaining = total - idx

            # Try (3,4) pattern → pos + quat
            if remaining >= 7:
                pos = take(3, "vec3")
                quat = take(4, "quat")

                if quat is not None:
                    norm = jp.linalg.norm(quat)
                    print(f"  └─ quat_norm → {norm:.3f}")

            elif remaining >= 4:
                take(4, "vec4")

            elif remaining >= 3:
                take(3, "vec3")

            else:
                take(1, "scalar")

        print(f"\nTotal used: {idx} / {total}")
        print("=" * 70 + "\n")

    # Run for both state + privileged
    for k, v in obs_dict.items():
        debug_single(v, k)

In [ ]:
obs_debug_full(state.obs, env)

In [ ]:

if TRAINING:
    make_inference_fn, params, metrics = train_fn(
        environment=env,
        wrap_env_fn=wrapper.wrap_for_brax_training,
    )

    print(f"time to jit: {times[1] - times[0]}")
    print(f"time to train: {times[-1] - times[1]}")
else:
    # Load the latest brax checkpoint automatically
    from brax.training.agents.ppo import checkpoint as ppo_checkpoint
    from brax.training.agents.ppo import networks as ppo_networks
    from brax.training import checkpoint as brax_checkpoint, networks as brax_networks
    from pathlib import Path
    import json as _json

    checkpoint_dir = Path(f'./checkpoints/run_{CHECKPOINT_IDX}').resolve()
    # Find the latest checkpoint (highest step number)
    ckpt_dirs = sorted(
        [d for d in checkpoint_dir.iterdir() if d.is_dir() and d.name.isdigit()],
        key=lambda d: int(d.name)
    )
    latest_ckpt = ckpt_dirs[-1]
    print(f"Loading checkpoint: {latest_ckpt.name} (step {int(latest_ckpt.name):,})")

    # Load config manually, working around brax bug with null kernel init fns
    config_path = latest_ckpt / 'ppo_network_config.json'
    loaded_dict = _json.loads(config_path.read_text())
    nf_kwargs = loaded_dict['network_factory_kwargs']
    if 'activation' in nf_kwargs:
        nf_kwargs['activation'] = brax_networks.ACTIVATION[nf_kwargs['activation']]
    for k in ('policy_network_kernel_init_fn', 'value_network_kernel_init_fn',
              'q_network_kernel_init_fn', 'mean_kernel_init_fn'):
        if k in nf_kwargs:
            if nf_kwargs[k] is None:
                del nf_kwargs[k]
            else:
                nf_kwargs[k] = brax_networks.KERNEL_INITIALIZER[nf_kwargs[k]]
    nf_kwargs = {k: v for k, v in nf_kwargs.items() if v is not None}

    # Fix observation_size: convert {"state": {"shape": [52], ...}} -> {"state": 52, ...}
    obs_size_raw = loaded_dict['observation_size']
    if isinstance(obs_size_raw, dict):
        obs_size = {}
        for key, val in obs_size_raw.items():
            if isinstance(val, dict) and 'shape' in val:
                shape = val['shape']
                obs_size[key] = shape[0] if len(shape) == 1 else tuple(shape)
            else:
                obs_size[key] = val
    else:
        obs_size = obs_size_raw

    normalize_observations = loaded_dict.get('normalize_observations', True)
    action_size = loaded_dict['action_size']

    # Load params
    params = ppo_checkpoint.load(str(latest_ckpt))

    # Rebuild network from config — use plain dicts, not ConfigDict
    ppo_network = ppo_networks.make_ppo_networks(
        observation_size=obs_size,
        action_size=action_size,
        preprocess_observations_fn=(
            brax_checkpoint.running_statistics.normalize
            if normalize_observations
            else lambda x, y: x
        ),
        **nf_kwargs,
    )
    make_inference_fn = ppo_networks.make_inference_fn(ppo_network)
    inference_fn = make_inference_fn(params, deterministic=True)
    jit_inference_fn = jax.jit(inference_fn)
    print("Policy loaded successfully.")

In [ ]:
# # Re-initialize the checkpoint manager, pointing to the same directory
# checkpoint_dir = os.path.abspath('./checkpoints')
# manager = ocp.CheckpointManager(
#     checkpoint_dir,
#     options=ocp.CheckpointManagerOptions(max_to_keep=1)
# )

# # Load checkpoint without providing a reference structure,
# # letting orbax infer shapes/dtypes from checkpoint metadata.
# restored_params = manager.restore(0)

# print("Parameters loaded successfully.")
# print("Restored type:", type(restored_params))

## Visualize Rollouts

In [ ]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
if TRAINING:
    jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

In [ ]:
!apt install -y ffmpeg

In [ ]:
rng = jax.random.PRNGKey(42)
rollout = []
n_episodes = 5

for _ in range(n_episodes):
  state = jit_reset(rng)
  rollout.append(state)
  for i in range(env_cfg.episode_length):
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)
    rollout.append(state)

render_every = 1
frames = env.render(rollout[::render_every])
rewards = [s.reward for s in rollout]
media.show_video(frames, fps=1.0 / env.dt / render_every)

In [ ]:
# rewards = jp.stack([s.reward for s in rollout])    # (T,)


# --- State-level fields ---
rewards = jp.stack([s.reward for s in rollout])          # (T,)
dones = jp.stack([s.done for s in rollout])              # (T,)
obs_all = jp.stack([s.obs['state'] for s in rollout])             # (T, obs_dim)

# --- MJX Data fields (physics state) ---
qpos = jp.stack([s.data.qpos for s in rollout])          # (T, nq) - joint positions
qvel = jp.stack([s.data.qvel for s in rollout])          # (T, nv) - joint velocities
ctrl = jp.stack([s.data.ctrl for s in rollout])           # (T, nu) - applied controls
xpos = jp.stack([s.data.xpos for s in rollout])          # (T, nbody, 3) - body positions
xquat = jp.stack([s.data.xquat for s in rollout])        # (T, nbody, 4) - body orientations
site_xpos = jp.stack([s.data.site_xpos for s in rollout])  # (T, nsite, 3) - site positions
mocap_pos = jp.stack([s.data.mocap_pos for s in rollout])   # (T, nmocap, 3) - mocap positions
mocap_quat = jp.stack([s.data.mocap_quat for s in rollout]) # (T, nmocap, 4) - mocap orientations
qfrc_actuator = jp.stack([s.data.qfrc_actuator for s in rollout])  # (T, nv) - actuator forces
sensordata = jp.stack([s.data.sensordata for s in rollout])  # (T, nsensor) - sensor readings
xfrc_applied = jp.stack([s.data.xfrc_applied for s in rollout])  # (T, nbody, 6) - applied forces

# --- Info dict (env-specific) ---
steps = jp.stack([s.info['_steps'] for s in rollout])    # (T,)

print(f"Rollout length: {len(rollout)}")
print(f"qpos: {qpos.shape}, qvel: {qvel.shape}, ctrl: {ctrl.shape}")
print(f"xpos: {xpos.shape}, xquat: {xquat.shape}")
print(f"sensordata: {sensordata.shape}")
print(f"mocap_pos: {mocap_pos.shape}")
print(f"mocap_quat: {mocap_quat.shape}")
print(f"rewards: {rewards.shape}, obs: {obs_all.shape}")

In [ ]:
plt.plot(rewards)
plt.show()

In [ ]:

plt.plot(qvel)
plt.show()

While the above policy is very simple, the work was extended using the Madrona batch renderer, and policies were transferred on a real robot. We encourage folks to check out the Madrona-MJX tutorial notebooks ([part 1](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_1.ipynb) and [part 2](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_2.ipynb))!

# Dexterous Manipulation

Let's now train a policy that was transferred onto a real Leap Hand robot with the `LeapCubeReorient` environment! The environment contains a cube placed in the center of the hand, and the goal is to re-orient the cube in SO(3).

In [ ]:
env_name = 'LeapCubeReorient'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)

In [ ]:
env_cfg

## Train Policy

Let's train an initial policy and visualize the rollouts. Notice that the PPO parameters contain `policy_obs_key` and `value_obs_key` fields, which allow us to train brax PPO with [asymmetric](https://arxiv.org/abs/1710.06542) observations for the actor and the critic. While the actor recieves proprioceptive state similar in nature to the real-world camera tracking sensors, the critic network recieves privileged state only available in the simulator. This enables more sample efficient learning, and we are able to train an initial policy in 33 minutes on a single RTX 4090.

Depending on the GPU device and topology, training can be brought down to 10-20 minutes as shown in the MuJoCo Playground technical report.

### PPO

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **ppo_params.network_factory
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    seed=1
)

In [ ]:
make_inference_fn, params, metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

## Visualize Rollouts

In [ ]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

In [ ]:
rng = jax.random.PRNGKey(42)
rollout = []
n_episodes = 1

for _ in range(n_episodes):
  state = jit_reset(rng)
  rollout.append(state)
  for i in range(env_cfg.episode_length):
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)
    rollout.append(state)

render_every = 1
frames = env.render(rollout[::render_every])
rewards = [s.reward for s in rollout]
media.show_video(frames, fps=1.0 / env.dt / render_every)

The above policy solves the task, but may look a little bit jittery. To get robust sim-to-real transfer,  we retrained from previous checkpoints using a curriculum on the maximum torque to facilitate exploration early on in the curriculum, and to produce smoother actions for the final policy. More details can be found in the MuJoCo Playground technical report!

🙌 Thanks for stopping by The Playground!